# Opening Outcome Prediction

This notebook presents two XGBoost outcome models trained from the KnightVision `silver_games` analytical view.

- `pre_game`: honest prediction using information known before the game starts.
- `post_game`: diagnostic comparison that also uses move metadata observed after the game.

Only the pre-game model should be used for prediction claims.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import Image, Markdown, display
except ImportError:
    class Markdown(str):
        pass

    class Image:
        def __init__(self, filename: str):
            self.filename = filename

        def __repr__(self) -> str:
            return f"Image({self.filename})"

    def display(value) -> None:
        print(value)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "models").exists():
            return candidate
    return current.parent if current.name == "notebooks" else current


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def read_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def require_artifacts(paths: list[Path]) -> None:
    missing = [path for path in paths if not path.exists()]
    if missing:
        display(Markdown("### Missing artifacts"))
        display(Markdown("Run the matching `make train-*` target, then rerun this notebook."))
        for path in missing:
            display(Markdown(f"- `{path.relative_to(REPO_ROOT)}`"))
        raise FileNotFoundError("Required ML artifacts are missing")


def metric_frame(metrics: dict[str, float]) -> pd.DataFrame:
    return (
        pd.Series(metrics, name="value")
        .rename_axis("metric")
        .reset_index()
        .assign(value=lambda frame: frame["value"].map(lambda value: f"{float(value):.4f}"))
    )


def display_markdown_file(path: Path) -> None:
    display(Markdown(path.read_text(encoding="utf-8")))


print(f"Repo root: {REPO_ROOT}")

ARTIFACT_DIR = REPO_ROOT / "models" / "opening_outcome"
required = [
    ARTIFACT_DIR / "metrics.json",
    ARTIFACT_DIR / "comparison_metrics.csv",
    ARTIFACT_DIR / "comparison_report.md",
    ARTIFACT_DIR / "pre_game" / "confusion_matrix.png",
    ARTIFACT_DIR / "pre_game" / "feature_importance.png",
    ARTIFACT_DIR / "pre_game" / "per_class_f1.png",
    ARTIFACT_DIR / "post_game" / "confusion_matrix.png",
    ARTIFACT_DIR / "post_game" / "feature_importance.png",
    ARTIFACT_DIR / "post_game" / "per_class_f1.png",
]
require_artifacts(required)
metadata = read_json(ARTIFACT_DIR / "metrics.json")
metadata["models"].keys()


## Dataset And Target

The target has three classes: `white_win`, `black_win`, and `draw`. The benchmark slice is large enough for training, but draw rows are much rarer than decisive results, so balanced accuracy and macro F1 matter more than raw accuracy alone.


In [ ]:
pre_game = metadata["models"]["pre_game"]
class_counts = pd.Series(pre_game["class_counts"], name="rows").rename_axis("result").reset_index()
class_counts["share"] = class_counts["rows"] / class_counts["rows"].sum()
class_counts


## Feature Sets

The pre-game model uses Elo, opening, time control, and date fields. The post-game diagnostic model adds parsed move metadata such as game length, captures, castling flags, clock availability, and legal prefix length.


In [ ]:
feature_rows = []
for model_name, model_metadata in metadata["models"].items():
    for feature_type in ["numeric", "categorical"]:
        for feature in model_metadata["features"][feature_type]:
            feature_rows.append({"model": model_name, "feature_type": feature_type, "feature": feature})
pd.DataFrame(feature_rows)


## Model Comparison

The pre-game model is the honest predictor. The post-game model is useful as a diagnostic ceiling because it can use information that was unavailable before the game started.


In [ ]:
comparison = pd.read_csv(ARTIFACT_DIR / "comparison_metrics.csv")
comparison.style.format({
    "accuracy": "{:.4f}",
    "balanced_accuracy": "{:.4f}",
    "macro_f1": "{:.4f}",
    "weighted_f1": "{:.4f}",
    "log_loss": "{:.4f}",
})


## Baseline Context

The model should be compared against simple policies: always choosing the majority class, using class-prior probabilities, and predicting the Elo favorite. This is especially important because outcome prediction before a chess game is inherently noisy.


In [ ]:
baseline_rows = []
for model_name, model_metadata in metadata["models"].items():
    baseline_rows.append({"model": model_name, "baseline": "xgboost", **model_metadata["metrics"]})
    for baseline_name, metrics in model_metadata["baselines"].items():
        baseline_rows.append({"model": model_name, "baseline": baseline_name, **metrics})
base = pd.DataFrame(baseline_rows)
base[["model", "baseline", "accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "log_loss"]].style.format({
    "accuracy": "{:.4f}",
    "balanced_accuracy": "{:.4f}",
    "macro_f1": "{:.4f}",
    "weighted_f1": "{:.4f}",
    "log_loss": "{:.4f}",
})


## Pre-Game Model Diagnostics

Use these charts for the model that can make legitimate pre-game predictions.


In [ ]:
PRE_GAME_DIR = ARTIFACT_DIR / "pre_game"
display(Image(filename=str(PRE_GAME_DIR / "confusion_matrix.png")))
display(Image(filename=str(PRE_GAME_DIR / "per_class_f1.png")))
display(Image(filename=str(PRE_GAME_DIR / "feature_importance.png")))


In [ ]:
pd.read_csv(PRE_GAME_DIR / "feature_importance.csv").head(20)


## Post-Game Diagnostic Model

This model uses after-the-game move features. It should be interpreted as a diagnostic comparison, not a fair predictor.


In [ ]:
POST_GAME_DIR = ARTIFACT_DIR / "post_game"
display(Image(filename=str(POST_GAME_DIR / "confusion_matrix.png")))
display(Image(filename=str(POST_GAME_DIR / "per_class_f1.png")))
display(Image(filename=str(POST_GAME_DIR / "feature_importance.png")))


In [ ]:
pd.read_csv(POST_GAME_DIR / "feature_importance.csv").head(20)


## Report And Interpretation

The comparison report records the key point: pre-game outcome prediction has weak signal, while post-game diagnostics are much stronger because they use information from the completed game.


In [ ]:
display_markdown_file(ARTIFACT_DIR / "comparison_report.md")


In [ ]:
display_markdown_file(PRE_GAME_DIR / "model_card.md")


## Optional Retraining

Set `RUN_TRAINING = True` only when the benchmark DuckDB warehouse exists and you want to regenerate artifacts.


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    from ml.opening_outcome.train import TrainingConfig, train

    metadata = train(
        TrainingConfig(
            duckdb_path=REPO_ROOT / "warehouse" / "knightvision_benchmark.duckdb",
            output_dir=ARTIFACT_DIR,
        )
    )
    metadata["models"].keys()
else:
    print("Skipping training. Set RUN_TRAINING = True to regenerate artifacts.")
